# ControlPlane.ai — the banking pilot, standalone

Runs `scripts/13_pilot_run.py` **only**, against reference activations attached
as a dataset instead of re-extracted. Twenty minutes rather than three hours.

**Attach the dataset before running:** `aditya103856/controlplane-reference-caches`
(Add Input → Datasets). It carries `cache-triviaqa-600.npz` and the eval sets
the reference probe is fitted on, under a manifest of SHA-256s and content
hashes.

Nothing here re-extracts. If the caches are wrong, the run refuses rather than
recomputing them — recomputing is the three-hour path this notebook exists to
avoid, and it should be a deliberate choice, not a fallback.

In [ ]:
import subprocess, sys, os
from pathlib import Path

CLONE_URL = "https://github.com/Aditya26189/controlplane.git"
CLONE_BRANCH = "main"

os.chdir("/kaggle/working")
if not Path("controlplane").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", CLONE_BRANCH,
                    CLONE_URL, "controlplane"], check=True)
os.chdir("/kaggle/working/controlplane")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True,
                     text=True).stdout.strip())

In [ ]:
!pip install -q bitsandbytes accelerate 2>&1 | tail -2

## 1 — Attach the caches, and check them

The manifest is verified against this checkout. A cache built from a different
eval set would still load and still produce numbers; only the content hash
catches that, and it is the same argument as the pilot's own draft-hash guard.

In [ ]:
import hashlib, json, shutil, sys
from pathlib import Path

sys.path.insert(0, "/kaggle/working/controlplane")

# Do not assume the mount path. Kaggle derives the input directory from the
# dataset's slug, and version 1 of this kernel died on a hardcoded
# /kaggle/input/controlplane-reference-caches that did not exist -- with the
# metadata correctly listing the dataset. Find the manifest instead, and print
# what IS mounted so the next failure diagnoses itself.
INPUT = Path("/kaggle/input")
# rglob, not glob: Kaggle mounted this dataset at
# /kaggle/input/datasets/<owner>/<slug>/, not the /kaggle/input/<slug>/
# the docs imply. The depth is not worth predicting -- search for the
# manifest wherever it landed.
candidates = sorted(INPUT.rglob("MANIFEST.json")) if INPUT.exists() else []
if not candidates:
    print("nothing mounted under /kaggle/input carries a MANIFEST.json.")
    print("what IS mounted:")
    if INPUT.exists():
        for entry in sorted(INPUT.rglob("*"))[:40]:
            print("   ", entry.relative_to(INPUT))
    else:
        print("    (/kaggle/input does not exist)")
    raise SystemExit(
        "Add Input -> Datasets -> "
        "aditya103856/controlplane-reference-caches, then re-run. Refusing to "
        "re-extract: that is the three-hour path this notebook replaces."
    )
SOURCE = candidates[0].parent
print("reference caches mounted at", SOURCE)

manifest = json.loads((SOURCE / "MANIFEST.json").read_text(encoding="utf-8"))
print("caches built from commit", manifest["built_from_commit"][:8])

Path("results").mkdir(exist_ok=True)
Path("evalsets").mkdir(exist_ok=True)
for item in sorted(SOURCE.iterdir()):
    if item.suffix == ".npz":
        shutil.copy2(item, Path("results") / item.name)
    elif item.suffix == ".json" and item.name != "MANIFEST.json":
        shutil.copy2(item, Path("evalsets") / item.name)

# SHA-256 of every cache, against the manifest. Cheap, and the alternative is
# discovering a truncated upload after the model has loaded.
for name, meta in manifest["files"].items():
    digest = hashlib.sha256((Path("results") / name).read_bytes()).hexdigest()
    if digest != meta["sha256"]:
        raise SystemExit(f"{name} sha256 {digest} != manifest {meta['sha256']}")
    print(f"  {name:<34} {meta['bytes'] / 2**20:7.1f} MiB  sha256 ok")

# The eval sets must be the ones THIS checkout believes in, or the reference
# scores describe a different envelope than the warrant names.
from controlplane.evalsets.registry import load_evalset

for eval_set_id, meta in manifest["evalsets"].items():
    local = Path("evalsets") / f"{eval_set_id}.json"
    if not local.exists():
        continue
    got = load_evalset(local).content_hash
    if got != meta["content_hash"]:
        raise SystemExit(
            f"{eval_set_id}: content hash {got[:16]} != manifest "
            f"{meta['content_hash'][:16]}. The cached activations were "
            "extracted from a different eval set."
        )
    print(f"  {eval_set_id:<34} content hash ok ({meta['n_items']} items)")

## 2 — The GPU is untouched

No extraction ran in this kernel, so the card should be empty. Checked anyway,
because the assumption that it was free is what cost the 2026-08-30 run: the
full notebook held 11.00 GiB of a 14.56 GiB card in the kernel process while
the pilot tried to load its own copy in a subprocess.

In [ ]:
import torch

free = {i: torch.cuda.mem_get_info(i)[0] / 2**30 for i in range(torch.cuda.device_count())}
print({i: f"{g:.1f} GiB free" for i, g in free.items()})

NEEDED_GIB = 6.0
if not free:
    raise SystemExit("no GPU visible; the pilot generates and extracts and needs one")
if max(free.values()) < NEEDED_GIB:
    raise SystemExit(
        f"only {max(free.values()):.1f} GiB free; the pilot needs about "
        f"{NEEDED_GIB} GiB. Restart the kernel."
    )

## 3 — The pilot

Generates 24 answers, judges them against gold aliases, checks `101`'s
acceptance band **before** scoring anything, extracts question-time
activations, and computes the IQR ratio against the reference envelope.

It reports a branch; it does not take one. Its first act is to verify that the
draft it rebuilt from `BANKING_PILOT_QUESTIONS` matches the committed freeze —
that guard passed on the 2026-08-30 run at hash `312516ded744e4fe`, before the
OOM killed the stage after it.

In [ ]:
import subprocess, sys

stage = ["scripts/13_pilot_run.py", "--config", "config.yaml",
         "--cache", "results/cache-triviaqa-600.npz"]
print(">>>", " ".join(stage))
done = subprocess.run([sys.executable, *stage], text=True, capture_output=True)
print(done.stdout[-8000:])
if done.returncode != 0:
    print(done.stderr[-8000:])
    raise SystemExit("pilot failed")

In [ ]:
from pathlib import Path

for name in ("pilot_run.json", "pilot_envelope.json"):
    path = Path("results") / name
    print("=" * 70)
    print(name, "--", "present" if path.exists() else "ABSENT")
    if path.exists():
        print(path.read_text(encoding="utf-8")[:3000])